# 03 — Machine Learning Modelling

Interactive ML exploration. Run `build_ml_dataset.py` and `train_ml_models.py` first for the full pipeline. This notebook provides interactive model analysis.

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_curve, roc_auc_score

sns.set_style('whitegrid')
%matplotlib inline

In [ ]:
# Load ML dataset
ml_path = Path('../data/features/ml_dataset.csv')
if not ml_path.exists():
    print('Run build_ml_dataset.py first')
else:
    df = pd.read_csv(ml_path)
    print(f'Shape: {df.shape}')
    
    # Identify label
    label_candidates = ['OS_class', 'Cluster', 'Progression', 'Response']
    label_col = next((c for c in label_candidates if c in df.columns), None)
    print(f'Label column: {label_col}')
    print(f'Label distribution:\n{df[label_col].value_counts()}')
    
    meta_cols = ['PatientID', 'Timepoint', label_col]
    feat_cols = [c for c in df.columns if c not in meta_cols]
    X = df[feat_cols]
    y = df[label_col].astype(int).values
    print(f'\nFeature matrix: {X.shape}')

In [ ]:
# Feature correlation heatmap (top 30 by variance)
top_var = X.var().nlargest(30).index
fig, ax = plt.subplots(figsize=(12, 10))
corr = X[top_var].corr(method='spearman')
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap='RdBu_r', center=0,
            square=True, linewidths=0.3, ax=ax, cbar_kws={'shrink': 0.7})
ax.set_xticklabels([c.split('_')[-1][:15] for c in top_var], rotation=90, fontsize=7)
ax.set_yticklabels([c.split('_')[-1][:15] for c in top_var], rotation=0, fontsize=7)
ax.set_title('Spearman Correlation — Top 30 Features by Variance')
plt.tight_layout()
plt.savefig('../results/figures/feature_correlation.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Quick 5-fold CV comparison: volume-only vs all radiomics
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

experiments = {}
vol_cols = [c for c in feat_cols if 'shape_' in c]
experiments['Volume Only'] = (X[vol_cols] if vol_cols else X.iloc[:, :5], y)
experiments['All Radiomics'] = (X, y)

fig, ax = plt.subplots(figsize=(7, 6))
for name, (X_exp, y_exp) in experiments.items():
    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(penalty='l2', C=0.1, class_weight='balanced',
                                    random_state=42, max_iter=500))
    ])
    y_prob = cross_val_predict(pipe, X_exp, y_exp, cv=skf, method='predict_proba')[:, 1]
    fpr, tpr, _ = roc_curve(y_exp, y_prob)
    auc = roc_auc_score(y_exp, y_prob)
    ax.plot(fpr, tpr, lw=2, label=f'{name} (AUC={auc:.3f})')

ax.plot([0,1],[0,1],'k--', lw=0.8)
ax.set_xlabel('1 – Specificity')
ax.set_ylabel('Sensitivity')
ax.set_title('ROC Curves (Logistic Regression, 5-fold CV)')
ax.legend()
plt.tight_layout()
plt.savefig('../results/figures/roc_curves_notebook.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Show pre-computed model comparison table if available
table_path = Path('../results/tables/model_comparison.csv')
if table_path.exists():
    df_res = pd.read_csv(table_path)
    display(df_res[['Experiment','Model','AUC','AUC_CI_lo','AUC_CI_hi',
                     'BalancedAcc','F1','Sensitivity','Specificity']]
              .sort_values('AUC', ascending=False).reset_index(drop=True))